# 2A · Tarea A: intención

Práctica final · **Asistente de agenda en lenguaje natural** · Procesamiento del Lenguaje Natural

Poned en la misma carpeta este notebook, `utilidades.py`, `evaluar.py`, `train.jsonl` y `agenda.jsonl`. Las celdas marcadas con `# TODO` son las que tenéis que completar.

Cada enfoque es una función `predecir(textos) -> lista de intenciones`. Las plantillas devuelven `"otro"` para que el notebook se ejecute desde el principio.

In [ ]:
import time, json
from utilidades import cargar, guardar, particiones, INTENCIONES, TIPOS, HOY
from evaluar import evaluar, imprimir

train, val = particiones()
agenda = cargar("agenda.jsonl")
X_train, X_val = [x["texto"] for x in train], [x["texto"] for x in val]
RESULTADOS = {}      # enfoque -> métricas y predicciones en validación

def registrar(nombre, preds, segundos):
    r = evaluar(val, preds)
    rd = evaluar(val, preds, "dificil") if any(x["dificil"] for x in val) else {}   # hasta que tengáis frases difíciles
    RESULTADOS[nombre] = {"metrica": r["A_f1_macro"], "dificiles": rd.get("A_f1_macro"),
                          "ms_por_frase": 1000 * segundos / max(len(preds), 1), "preds": preds}
    print(f"{nombre:<40} A_f1_macro = {r['A_f1_macro']:.3f}   difíciles = {rd.get('A_f1_macro', '—')}")

In [ ]:
def preds_A(etiquetas):
    return [{"id": x["id"], "intencion": e} for x, e in zip(val, etiquetas)]

### A1 · Reglas
Palabras clave y expresiones regulares (temas 1–2).

In [ ]:
import re
def intencion_reglas(texto):
    # TODO: reglas ordenadas (borrar, completar, modificar, buscar, consultar, crear_tarea, crear_evento)
    return "otro"
t0 = time.time(); registrar("A1 reglas", preds_A([intencion_reglas(t) for t in X_val]), time.time() - t0)

### A2 · Bolsa de palabras + Naive Bayes
Tema 4.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
# TODO: modelo = ... .fit(X_train, y_train); pred = modelo.predict(X_val)
pred = ["otro"] * len(X_val)
registrar("A2 BoW + Naive Bayes", preds_A(pred), 0)

### A3 · TF-IDF + regresión logística
Palabras y n-gramas de caracteres (tema 4).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
pred = ["otro"] * len(X_val)          # TODO
registrar("A3 TF-IDF + RL", preds_A(pred), 0)

### A4 · Word2Vec propio (media de vectores) + clasificador
Entrenadlo con gensim sobre vuestro corpus (tema 4).

In [ ]:
# from gensim.models import Word2Vec
pred = ["otro"] * len(X_val)          # TODO
registrar("A4 Word2Vec propio + clasificador", preds_A(pred), 0)

### A5 · Embeddings estáticos preentrenados
Vectores de spaCy o FastText en español.

In [ ]:
pred = ["otro"] * len(X_val)          # TODO
registrar("A5 vectores preentrenados + clasificador", preds_A(pred), 0)

### A6 · Embeddings de frase + regresión logística
El modelo **no se entrena**: calculad los vectores una vez y reutilizadlos en A7 (tema 4, parte 2).

In [ ]:
# from sentence_transformers import SentenceTransformer
# m = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# E_train, E_val = m.encode(X_train), m.encode(X_val)
pred = ["otro"] * len(X_val)          # TODO
registrar("A6 embeddings + RL", preds_A(pred), 0)

### A7 · Los mismos embeddings + Random Forest y MLP
¿Qué clasificador aprovecha mejor los embeddings?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
for nombre in ["A7 embeddings + Random Forest", "A7 embeddings + MLP"]:
    pred = ["otro"] * len(X_val)      # TODO
    registrar(nombre, preds_A(pred), 0)

### A8 · LLM sin entrenar: zero-shot y few-shot
Convertid la respuesta en una de las 8 intenciones y contad las inválidas. Usad al menos un modelo de 1,5B.

In [ ]:
# from transformers import pipeline
# llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")
PROMPT = "TODO: instrucciones + (few-shot) ejemplos + la frase"
def intencion_llm(texto):
    # TODO: llamar al LLM y convertir su respuesta en una intención válida (o "otro")
    return "otro"
for nombre in ["A8 LLM zero-shot", "A8 LLM few-shot"]:
    t0 = time.time(); registrar(nombre, preds_A([intencion_llm(t) for t in X_val]), time.time() - t0)

> Para los experimentos E1 y E2 (notebook 3), añadid también a `tarea_a.py` una función `entrenar_y_predecir(enfoque, textos_train, etiquetas_train, textos_val)` para A2, A3, A6 y A7.

## Resumen y enfoque elegido
Guardamos los resultados de validación para el notebook 3 y elegimos un enfoque.

In [ ]:
import pandas as pd
print(pd.DataFrame({k: {c: v[c] for c in ("metrica", "dificiles", "ms_por_frase")} for k, v in RESULTADOS.items()}).T.round(3))
json.dump(RESULTADOS, open("resultados_A.json", "w", encoding="utf-8"), ensure_ascii=False)
MEJOR = max(RESULTADOS, key=lambda k: RESULTADOS[k]["metrica"])     # TODO: elegid y justificad el enfoque (calidad, coste, latencia…)
print("Enfoque elegido:", MEJOR)

## Exportar el enfoque elegido
Copiad a **`tarea_a.py`** el enfoque elegido (ya entrenado o con lo necesario para entrenarlo) como una función `predecir_intencion(texto) -> intención`. La usan el asistente (notebook 4) y la función `generar_predicciones` del notebook 3, que es la que ejecutará el profesor con el **test oculto**.